## This notebook shows examples of doing normalization and compute deviations and probabilities

### getnv(): get current normative setting/environment (from visualFields package)

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import linregress

# vfseries has both s1-s54 (sensitivity) and td1-td54 (TD computed by R)
# So: normative_expected = sensitivity - TD  per location per exam
df = pd.read_csv(r"C:\Users\mm3572\PyVisualField\src\PyVisualFields\data\vfseries.csv")

print(df.columns.tolist())  # verify column names first

['eyeid', 'nvisit', 'yearsfollowed', 'distprev', 'age', 'righteye', 'duration', 'falsenegrate', 'falseposrate', 'malfixrate', 'ght', 'vfi', 'md', 'mdprob', 'psd', 'psdprob', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21', 's22', 's23', 's24', 's25', 's26', 's27', 's28', 's29', 's30', 's31', 's32', 's33', 's34', 's35', 's36', 's37', 's38', 's39', 's40', 's41', 's42', 's43', 's44', 's45', 's46', 's47', 's48', 's49', 's50', 's51', 's52', 's53', 's54', 'td1', 'td2', 'td3', 'td4', 'td5', 'td6', 'td7', 'td8', 'td9', 'td10', 'td11', 'td12', 'td13', 'td14', 'td15', 'td16', 'td17', 'td18', 'td19', 'td20', 'td21', 'td22', 'td23', 'td24', 'td25', 'td27', 'td28', 'td29', 'td30', 'td31', 'td32', 'td33', 'td34', 'td36', 'td37', 'td38', 'td39', 'td40', 'td41', 'td42', 'td43', 'td44', 'td45', 'td46', 'td47', 'td48', 'td49', 'td50', 'td51', 'td52', 'td53', 'td54', 'pd1', 'pd2', 'pd3', 'pd4', 'pd5', 'pd6', 'pd7', '

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import linregress

# Load control dataset
df = pd.read_csv(r"C:\Users\mm3572\PyVisualField\src\PyVisualFields\data\vfctrSunyiu24d2.csv")

# Sensitivity columns
scols = sorted([c for c in df.columns if c.startswith('l') and c[1:].isdigit()],
               key=lambda x: int(x[1:]))
ages = df['age'].values.astype(float)

# R's agelm uses INVERSE FREQUENCY weights:
# each patient's visits get weight = 1/(n_visits_for_that_patient)
# normalized by n_unique_patients
ids = df['id'].values
unique_ids, counts = np.unique(ids, return_counts=True)
id_to_count = dict(zip(unique_ids, counts))
n_patients = len(unique_ids)

# Weight per row: 1/n_visits_for_this_patient / n_patients
weights = np.array([1.0 / (id_to_count[i] * n_patients) for i in ids])

intercepts, slopes = [], []
for col in scols:
    y = df[col].values.astype(float)
    mask = ~np.isnan(y)
    y_m = y[mask]; a_m = ages[mask]; w_m = weights[mask]
    w_m = w_m / w_m.sum()  # normalize weights

    # Weighted OLS: same as R's lm(y ~ age, weights=w)
    x_m  = np.column_stack([np.ones(len(a_m)), a_m])
    W    = np.diag(w_m)
    beta = np.linalg.lstsq(x_m.T @ W @ x_m, x_m.T @ W @ y_m, rcond=None)[0]
    intercepts.append(beta[0])
    slopes.append(beta[1])

out = r"C:\Users\mm3572\PyVisualField\src\PyVisualFields\data\normvals_sunyiu24d2.csv"
pd.DataFrame({'intercept': intercepts, 'slope': slopes}).to_csv(out, index=False)
print(f"Saved {len(intercepts)} weighted normative points")
print("First 5 intercepts:", np.round(intercepts[:5], 3))
print("First 5 slopes:    ", np.round(slopes[:5], 5))

Saved 54 weighted normative points
First 5 intercepts: [31.247 30.61  28.927 29.629 31.245]
First 5 slopes:     [-0.07557 -0.06485 -0.04759 -0.06163 -0.05061]


In [2]:
from PyVisualFields import visualFields

######## Get the current normalization information
currentNV = visualFields.getnv()

Current normative: vf_core default (Heijl 1987 24-2).


### Get all Normalized Computations
### Compute all td, pd, pdp, tdp, gl, gh,glp 
### From the input VF sensitivty based on the current NV

In [3]:
from PyVisualFields import visualFields

df_VFs_py = visualFields.data_vfpwgRetest24d2()
df_td, df_tdp, df_gi, df_gip, df_pd, df_pdp, gh = visualFields.getallvalues(df_VFs_py) #Input: dataframe includes sensitivity compatible with the format required by visualFields R package
a=print(df_td)

     id eye       date      time  age type  fpr  fnr    fl  duration  ...  \
0     1  OD 2008-08-13  00:00:00   53  pwg    0  0.0  0.00  00:00:00  ...   
1     1  OD 2008-08-20  00:00:00   53  pwg    0  0.0  0.06  00:00:00  ...   
2     1  OD 2008-08-27  00:00:00   53  pwg    0  0.0  0.00  00:00:00  ...   
3     1  OD 2008-09-03  00:00:00   53  pwg    0  0.0  0.06  00:00:00  ...   
4     1  OD 2008-09-10  00:00:00   53  pwg    0  0.0  0.13  00:00:00  ...   
..   ..  ..        ...       ...  ...  ...  ...  ...   ...       ...  ...   
355  30  OS 2009-05-08  00:00:00   68  pwg    0  0.0  0.00  00:00:00  ...   
356  30  OS 2009-05-15  00:00:00   68  pwg    0  0.0  0.00  00:00:00  ...   
357  30  OS 2009-05-25  00:00:00   68  pwg    0  0.0  0.00  00:00:00  ...   
358  30  OS 2009-05-29  00:00:00   68  pwg    0  0.0  0.00  00:00:00  ...   
359  30  OS 2009-06-05  00:00:00   68  pwg    0  0.0  0.07  00:00:00  ...   

         td45      td46      td47      td48      td49      td50      td51  

### Compute each of the td, pd, pdp, etc based on the current NV 
### Alias to the getallvalues
### Pay attention to the inputs

In [4]:
from PyVisualFields import visualFields

df_VFs_py = visualFields.data_vfpwgRetest24d2()

df_td = visualFields.gettd(df_VFs_py) # get TD values
df_gi = visualFields.getgl(df_VFs_py) # get global indices
df_tdp = visualFields.gettdp(df_td) # get TD probability values
df_pd = visualFields.getpd(df_td) # get PD values
gh = visualFields.getgh(df_td) # get the general height
df_pdp = visualFields.getpdp(df_pd) # get PD probability values
df_gip = visualFields.getglp(df_gi) # get global indices probability values

### Example to show required names of columns
### Based on vfprogression and visualFields packages, the names of columns of the input dataframe should be matched to function's requirment

In [5]:
from PyVisualFields import vfprogression
from PyVisualFields import visualFields
import pandas

df_VFs_py = vfprogression.data_vfseries()
columns2preserve = ['eyeid','righteye','age', 'duration', 
                     's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8',
                      's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16',
                       's17', 's18', 's19', 's20', 's21', 's22', 's23', 's24',
                        's25', 's26', 's27', 's28', 's29', 's30', 's31', 's32',
                         's33', 's34', 's35', 's36', 's37', 's38', 's39', 's40',
                         's41',  's42', 's43', 's44', 's45', 's46', 's47', 's48',
                          's49', 's50', 's51', 's52', 's53', 's54']
df_VFs_py = df_VFs_py[columns2preserve]
df_VFs_py.rename(columns={'s1':'l1', 's2':'l2', 's3':'l3', 
                                      's4':'l4', 's5':'l5', 's6':'l6', 's7':'l7',
                                      's8':'l8', 's9':'l9', 's10':'l10',
                                      's11':'l11', 's12':'l12', 's13':'l13',
                                      's14':'l14', 's15':'l15','s16':'l16',
                                      's17':'l17', 's18':'l18', 's19':'l19',
                      's20':'l20', 's21':'l21', 's22':'l22',
                      's23':'l23', 's24':'l24','s25':'l25', 's26':'l26', 
                      's27':'l27', 's28':'l28', 's29':'l29', 's30':'l30', 
                      's31':'l31', 's32':'l32','s33':'l33', 's34':'l34',
                      's35':'l35', 's36':'l36', 's37':'l37', 's38':'l38', 
                      's39':'l39', 's40':'l40', 's41':'l41', 
                         's42':'l42', 's43':'l43', 's44':'l44', 's45':'l45', 
                         's46':'l46', 's47':'l47', 's48':'l48',
                          's49':'l49', 's50':'l50', 's51':'l51', 's52':'l52',
                          's53':'l53', 's54':'l54'}, inplace=True)
df_VFs_py.rename(columns={"eyeid": "id"}, inplace=True)
df_VFs_py = df_VFs_py.replace({'righteye':{0:'OS', 1:'OD'}})
df_VFs_py.rename(columns={"righteye": "eye"}, inplace=True)
df_VFs_py.insert(1, "time", '00:00:00')
df_VFs_py.insert(1, 'date', 1000)
df_VFs_py['date'] = pandas.to_datetime(df_VFs_py['date'], errors='coerce')
df_VFs_py.insert(1, 'type', 'SIT')
df_VFs_py.insert(1, 'fpr', 0)
df_VFs_py.insert(1, 'fnr', 0)
df_VFs_py.insert(1, 'fl', 0)
df_td, df_tdp, df_gi, df_gip, df_pd, df_pdp, gh = visualFields.getallvalues(df_VFs_py)

C:\Users\mm3572\AppData\Local\anaconda3\envs\env_pyVF\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


### Get the all predefined normalization settings (from visualFields package)

In [6]:
from PyVisualFields import visualFields

NormValues = visualFields.normvals()
NormInfo = visualFields.get_info_normvals()

''' 
==> comment: > pw: pointwise, classic: smooth
==> comment: > default is: sunyiu_24d2
'''

Pure-Python mode: using vf_core normative DB.
Keys: default_24d2 (Heijl 1987 approximation)


' \n==> comment: > pw: pointwise, classic: smooth\n==> comment: > default is: sunyiu_24d2\n'

### set normalization values based on predefined ones (from visualFields package)

In [7]:
from PyVisualFields import visualFields

# we are going to change the setting from classic SUNY-IU to Iowa pointwise 
print('===> current NV setting:', )
currentNV = visualFields.getnv() 
target_predeifned_nv_name = 'iowa_Peri_pw'
visualFields.setnv(target_predeifned_nv_name)
print('===> new current NV setting:', )
currentNV = visualFields.getnv() # check it is set correctly:

===> current NV setting:
Current normative: vf_core default (Heijl 1987 24-2).
setnv: 'iowa_Peri_pw' — named normative sets not implemented in pure-Python mode.
===> new current NV setting:
Current normative: vf_core default (Heijl 1987 24-2).


### caculate new normalization values from a population and set it to be used:

In [8]:
from PyVisualFields import visualFields
import pickle

######## caculate new nv values and set it to be used:
df_VFs_py = visualFields.data_vfctrSunyiu24d2()
newNV_r, newNV_py = visualFields.nvgenerate(df_VFs_py, method = "pointwise",
                             name = "our_NV",
                             perimetry = "something",
                             strategy = "something",
                             size = "tmp")
visualFields.setnv(newNV_r)
print('===> current NV setting: ')
currentNV = visualFields.getnv() # check it is set correctly:

#''' notcie: this normalization will not be saved.
#We need to set for each session
#so we need to save and load them seperately, e.g.: '''
newNV_dict = { "newNV_r": newNV_r, "newNV_py": newNV_py }
pickle.dump( newNV_dict, open( "our_NV.pkl", "wb" ) )

loaded_dict = pickle.load( open( "our_NV.pkl", "rb" ) )
newNV_r = loaded_dict['newNV_r']
newNV_py = loaded_dict['newNV_py']
visualFields.setnv(newNV_r) # set it to be used

#### change the normalization values to the defalut of package: sunyiu_24d2
visualFields.setdefaults()

nvgenerate: fitted 54-point normative DB (method=pointwise).
setnv: custom normative DB set (54 points).
===> current NV setting: 
Current normative: custom DB (54 points).
setnv: custom normative DB set (54 points).
Defaults set: using vf_core built-in normative DB (Heijl 1987 24-2).
